import necessary libraries

In [1]:
import pandas as pd
import numpy as np

In [2]:
file = r"C:\Users\Lenovo\OneDrive\Documents\DataSprint\data\DataWave.csv"

df = pd.read_csv(file)


Inspect dataset


In [3]:
df.isnull().sum()

df.info()
df.describe()
df.head()
df.tail()
df.columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 705 entries, 0 to 704
Data columns (total 12 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   user_id                       695 non-null    object 
 1   country                       705 non-null    object 
 2   age                           705 non-null    int64  
 3   gender                        626 non-null    object 
 4   subscription_type             705 non-null    object 
 5   avg_listening_hours_per_week  705 non-null    float64
 6   total_songs_played            705 non-null    int64  
 7   skip_rate                     705 non-null    object 
 8   satisfaction_score            661 non-null    float64
 9   churned                       705 non-null    object 
 10  monthly_fee                   672 non-null    object 
 11  join_date                     705 non-null    object 
dtypes: float64(2), int64(2), object(8)
memory usage: 66.2+ KB


Index(['user_id', 'country', 'age', 'gender', 'subscription_type',
       'avg_listening_hours_per_week', 'total_songs_played', 'skip_rate',
       'satisfaction_score', 'churned', 'monthly_fee', 'join_date'],
      dtype='object')

cleaning user_id

In [4]:
df['user_id'].isnull().sum()
df = df.drop_duplicates(subset=['user_id'])

df= df.dropna(subset=['user_id'])


cleaned dataset

In [23]:
cleaned_dataset = r"C:\Users\Lenovo\OneDrive\Documents\DataSprint\data\cleaned_DataWave.csv"
df.to_csv(cleaned_dataset, index=False)

df = pd.read_csv(cleaned_dataset)

Convert all categorical text column to lower case and strip whitespaces. These are for the rows of each their respective columns. Ensure column is treated as text with the help of .astype(str).

In [6]:
cat_cols = ['country', 'gender', 'subscription_type', 'churned']
for col in cat_cols:
    df[col] = df[col].astype(str).str.strip().str.lower()

Cleaning gender values

In [7]:
df['gender'] = df['gender'].replace({
    'm': 'male',
    'f': 'female',
    'femle' : 'female',
    'female ': 'female',
    'others': 'other',
    'other ': 'other',
    'prefer not to say': 'unknown',
    'nan': np.nan
})

df['gender'].value_counts()   # Counting the number of each genders

gender
female    269
male      261
other      81
Name: count, dtype: int64

Cleaning churned columns. Convert yes/no or text to 0/1 integer

In [8]:
mapping = {
    'yes': 1, 'y': 1, '1': 1, 'true': 1,
    'no': 0, 'n': 0, '0': 0, 'false': 0
}

df['churned'] = (
    df['churned']
    .astype(str).str.strip().str.lower()
    .map(mapping)
)


df['churned'].mean()   # Finding the mean of churned

np.float64(0.30579710144927535)

Clean numeric columns. Convert invalid text like "ten" to NaN

In [9]:
num_cols = ['age', 'avg_listening_hours_per_week', 'total_songs_played', 'skip_rate', 'satisfaction_score', 'monthly_fee']

for cols in num_cols:
    df[cols] = pd.to_numeric(df[cols], errors='coerce')   # convert invalid values to NaN)


df['avg_listening_hours_per_week'].describe()   # Summary statistics of average listening hours per week

count    690.000000
mean      10.054203
std        4.804563
min        0.100000
25%        6.625000
50%       10.000000
75%       13.300000
max       23.700000
Name: avg_listening_hours_per_week, dtype: float64

OPTIONAL: 
1. Age should be between 10 and 100
2. Max listening hours per week = 168 (24*7)
3. Satisfaction must be between 1 and 10

In [10]:
df = df[(df['age'] >= 10) & (df['age'] <= 100)]
df = df[df['avg_listening_hours_per_week'] <= 168]   
df = df[(df['satisfaction_score'] >= 1) & (df['satisfaction_score'] <= 10)]

CLEAN join_date. Convert to datetime.

In [11]:
df['join_date'] = pd.to_datetime(df['join_date'], format='%m/%d/%Y', errors='coerce')
df['join_date'].isnull().sum()

np.int64(228)

Cleaning(Fix) Subscription Type

In [12]:
df['subscription_type'] = df['subscription_type'].replace({
    'premuim': 'premium',
    'premum': 'premium',
    'free trial': 'free',
    'fam': 'family',
    'stud': 'student',
    'studnt': 'student'
})

df['subscription_type'].value_counts()   # Counting the number of each subscription type

subscription_type
family     194
student    186
premium    171
free        95
Name: count, dtype: int64

Cleaning country

In [13]:
df['country'] = df['country'].replace({
    'u.k.': 'United kingdom',
    'uk': 'United Kingdom',
    'U.K': ' United Kingdom',
    'united kingdom': 'United Kingdom',
    'United kingdom': 'United Kingdom',
    'ind': 'India',
    'india': 'India',
    'usa': 'United States',
    'us': 'United States',
    'nepal': 'Nepal',
    'nigeria': 'Nigeria',
    'ghana': 'Ghana',
    'kenya': 'Kenya',
    'brazil': 'Brazil',
    'south africa': 'South Africa'
})
df['country'].value_counts()

country
United Kingdom    129
India             102
United States      72
Kenya              62
Nigeria            59
Ghana              52
South Africa       52
Nepal              41
United kingdom     39
Brazil             38
Name: count, dtype: int64

Checking Missing values

In [14]:
df.isnull().sum()

user_id                           0
country                           0
age                               0
gender                           74
subscription_type                 0
avg_listening_hours_per_week      0
total_songs_played                0
skip_rate                       503
satisfaction_score                0
churned                           0
monthly_fee                      72
join_date                       228
dtype: int64

Satisfaction score description.

In [15]:
df['satisfaction_score'].describe()

count    646.000000
mean       3.136223
std        1.186495
min        1.000000
25%        2.000000
50%        3.000000
75%        4.000000
max        5.000000
Name: satisfaction_score, dtype: float64

Relationship insights
1. Does churn differ by subscription type?(Churn rate by subscription type)

In [16]:
df.groupby('subscription_type')['churned'].mean().sort_values(ascending=False)

subscription_type
family     0.314433
student    0.306452
premium    0.304094
free       0.263158
Name: churned, dtype: float64

2. Do heavy listeners churn less? (Listening hours: churned vs active). Low hours = more churn, High hours = more engagement

In [17]:
df.groupby('churned')['avg_listening_hours_per_week'].mean()

churned
0    10.227494
1     9.889231
Name: avg_listening_hours_per_week, dtype: float64

3. Does satisfaction predict churn?(Satisfaction score: churned vs active). If churned users have lower scores then obvious red flag for company.

In [18]:
df.groupby('churned')['satisfaction_score'].mean()

churned
0    3.115299
1    3.184615
Name: satisfaction_score, dtype: float64

4. Which country or region has highest churn? (Top 10 countries with highest churn)

In [22]:
df.groupby('country')['churned'].mean().sort_values(ascending=False).head(10)

country
Brazil            0.368421
Ghana             0.365385
Nigeria           0.355932
United Kingdom    0.348837
Kenya             0.322581
South Africa      0.288462
United kingdom    0.282051
India             0.245098
United States     0.236111
Nepal             0.195122
Name: churned, dtype: float64